# Khả năng tổng quát hóa trong Phân loại (Generalization in Classification)

:label:`chap_classification_generalization`

Cho đến nay, chúng ta đã tập trung vào cách giải quyết các bài toán phân loại đa lớp bằng cách huấn luyện các mạng thần kinh (tuyến tính) với nhiều đầu ra và hàm softmax. Diễn giải các đầu ra của mô hình như các dự đoán xác suất, chúng ta đã thúc đẩy và xây dựng hàm mất mát entropy chéo (cross-entropy loss), hàm này tính toán log likelihood âm mà mô hình của chúng ta (cho một tập hợp các tham số cố định) gán cho các nhãn thực tế. Và cuối cùng, chúng ta đã đưa các công cụ này vào thực tế bằng cách khớp mô hình của mình với tập huấn luyện. Tuy nhiên, như mọi khi, mục tiêu của chúng ta là tìm hiểu *các quy luật tổng quát*, được đánh giá bằng thực nghiệm trên dữ liệu chưa từng thấy trước đây (tập kiểm thử - test set). Độ chính xác cao trên tập huấn luyện không có ý nghĩa gì cả. Bất cứ khi nào mỗi đầu vào của chúng ta là duy nhất (và thực tế điều này đúng với hầu hết các tập dữ liệu nhiều chiều), chúng ta có thể đạt được độ chính xác hoàn hảo trên tập huấn luyện bằng cách ghi nhớ tập dữ liệu ở chu kỳ huấn luyện (epoch) đầu tiên và sau đó tra cứu nhãn bất cứ khi nào chúng ta nhìn thấy một hình ảnh mới. Tuy nhiên, việc ghi nhớ các nhãn chính xác liên kết với các ví dụ huấn luyện chính xác không cho chúng ta biết cách phân loại các ví dụ mới. Nếu không có hướng dẫn nào thêm, chúng ta có thể phải dựa vào việc đoán ngẫu nhiên bất cứ khi nào gặp các ví dụ mới.

Một số câu hỏi hóc búa cần được chú ý ngay lập tức:

1. Chúng ta cần bao nhiêu ví dụ kiểm thử để đưa ra một ước lượng tốt về độ chính xác của bộ phân loại trên tổng thể bên dưới?
2. Điều gì xảy ra nếu chúng ta tiếp tục đánh giá lặp đi lặp lại các mô hình trên cùng một tập kiểm thử?
3. Tại sao chúng ta nên kỳ vọng rằng việc khớp các mô hình tuyến tính của mình với tập huấn luyện sẽ mang lại kết quả tốt hơn so với sơ đồ ghi nhớ ngây thơ?

Trong khi :numref:`sec_generalization_basics` đã giới thiệu những điều cơ bản về hiện tượng quá khớp (overfitting) và khả năng tổng quát hóa (generalization) trong bối cảnh hồi quy tuyến tính, chương này sẽ đi sâu hơn một chút, giới thiệu một số ý tưởng nền tảng của lý thuyết học thống kê. Hóa ra chúng ta thường có thể đảm bảo khả năng tổng quát hóa *a priori* (tiên nghiệm): đối với nhiều mô hình và đối với bất kỳ giới hạn trên mong muốn nào đối với khoảng cách tổng quát hóa (generalization gap) $\epsilon$, chúng ta thường có thể xác định một số lượng mẫu cần thiết $n$ sao cho nếu tập huấn luyện của chúng ta chứa ít nhất $n$ mẫu, sai số thực nghiệm của chúng ta sẽ nằm trong khoảng $\epsilon$ của sai số thực tế, *cho bất kỳ phân phối tạo dữ liệu nào*. Thật không may, cũng hóa ra là mặc dù các loại đảm bảo này cung cấp một tập hợp các khối xây dựng trí tuệ sâu sắc, chúng lại có tiện ích thực tế hạn chế đối với người thực hành học sâu. Nói ngắn gọn, các đảm bảo này gợi ý rằng việc đảm bảo khả năng tổng quát hóa của các mạng thần kinh sâu *a priori* đòi hỏi một số lượng ví dụ phi lý (có lẽ là hàng nghìn tỷ hoặc hơn), ngay cả khi chúng ta thấy rằng, đối với các nhiệm vụ chúng ta quan tâm, các mạng thần kinh sâu thường tổng quát hóa tốt một cách đáng kinh ngạc với ít ví dụ hơn nhiều (hàng nghìn). Do đó, những người thực hành học sâu thường từ bỏ hoàn toàn các đảm bảo *a priori*, thay vào đó áp dụng các phương pháp đã tổng quát hóa tốt trên các bài toán tương tự trong quá khứ và chứng nhận khả năng tổng quát hóa *post hoc* (hậu nghiệm) thông qua các đánh giá thực nghiệm. Khi chuyển sang :numref:`chap_perceptrons`, chúng ta sẽ xem xét lại khả năng tổng quát hóa và cung cấp một giới thiệu sơ lược về kho tàng tài liệu khoa học khổng lồ đã xuất hiện trong các nỗ lực giải thích lý do tại sao các mạng thần kinh sâu lại tổng quát hóa tốt trong thực tế.

## Tập Kiểm Thử (The Test Set)

Vì chúng ta đã bắt đầu dựa vào các tập kiểm thử như phương pháp tiêu chuẩn vàng để đánh giá sai số tổng quát hóa, hãy bắt đầu bằng việc thảo luận về các thuộc tính của các ước lượng sai số này. Hãy tập trung vào một bộ phân loại cố định $f$, mà không cần lo lắng về cách nó được tạo ra. Hơn nữa, giả sử chúng ta sở hữu một tập dữ liệu *mới* gồm các ví dụ $\mathcal{D} = {(\mathbf{x}^{(i)},y^{(i)})}_{i=1}^n$ không được sử dụng để huấn luyện bộ phân loại $f$. *Sai số thực nghiệm* (empirical error) của bộ phân loại $f$ trên $\mathcal{D}$ đơn giản là tỷ lệ các trường hợp mà dự đoán $f(\mathbf{x}^{(i)})$ không khớp với nhãn thực tế $y^{(i)}$, và được đưa ra bởi biểu thức sau:

$$\epsilon_\mathcal{D}(f) = \frac{1}{n}\sum_{i=1}^n \mathbf{1}(f(\mathbf{x}^{(i)}) \neq y^{(i)}).$$

Ngược lại, *sai số tổng thể* (population error) là tỷ lệ *kỳ vọng* của các ví dụ trong tổng thể bên dưới (một phân phối $P(X,Y)$ được đặc trưng bởi hàm mật độ xác suất $p(\mathbf{x},y)$) mà bộ phân loại của chúng ta không khớp với nhãn thực tế:

$$\epsilon(f) =  E_{(\mathbf{x}, y) \sim P} \mathbf{1}(f(\mathbf{x}) \neq y) = \int\int \mathbf{1}(f(\mathbf{x}) \neq y) p(\mathbf{x}, y) \;d\mathbf{x} dy.$$

Mặc dù $\epsilon(f)$ là đại lượng mà chúng ta thực sự quan tâm, chúng ta không thể quan sát trực tiếp nó, giống như chúng ta không thể trực tiếp quan sát chiều cao trung bình trong một tổng thể lớn mà không đo lường từng người một. Chúng ta chỉ có thể ước lượng đại lượng này dựa trên các mẫu. Bởi vì tập kiểm thử $\mathcal{D}$ của chúng ta mang tính đại diện thống kê cho tổng thể bên dưới, chúng ta có thể coi $\epsilon_\mathcal{D}(f)$ như một ước lượng thống kê của sai số tổng thể $\epsilon(f)$. Hơn nữa, bởi vì đại lượng quan tâm $\epsilon(f)$ là một kỳ vọng (của biến ngẫu nhiên $\mathbf{1}(f(X) \neq Y)$) và ước lượng tương ứng $\epsilon_\mathcal{D}(f)$ là trung bình mẫu, việc ước lượng sai số tổng thể chỉ đơn giản là bài toán ước lượng trung bình cổ điển mà bạn có thể nhớ lại từ :numref:`sec_prob`.

Một kết quả cổ điển quan trọng từ lý thuyết xác suất gọi là *định lý giới hạn trung tâm* đảm bảo rằng bất cứ khi nào chúng ta sở hữu $n$ mẫu ngẫu nhiên $a_1, ..., a_n$ được rút ra từ bất kỳ phân phối nào có trung bình $\mu$ và độ lệch chuẩn $\sigma$, thì khi số lượng mẫu $n$ tiến tới vô cùng, trung bình mẫu $\hat{\mu}$ xấp xỉ có xu hướng tiến tới phân phối chuẩn tập trung tại trung bình thực và có độ lệch chuẩn $\sigma/\sqrt{n}$. Điều này đã cho chúng ta biết một điều quan trọng: khi số lượng ví dụ tăng lên, sai số kiểm thử của chúng ta $\epsilon_\mathcal{D}(f)$ sẽ tiếp cận sai số thực tế $\epsilon(f)$ với tốc độ $\mathcal{O}(1/\sqrt{n})$. Do đó, để ước lượng sai số kiểm thử chính xác hơn gấp đôi, chúng ta phải thu thập tập kiểm thử lớn gấp bốn lần. Để giảm sai số kiểm thử đi một trăm lần, chúng ta phải thu thập tập kiểm thử lớn gấp mười nghìn lần. Nói chung, tốc độ $\mathcal{O}(1/\sqrt{n})$ này thường là mức tốt nhất mà chúng ta có thể hy vọng trong thống kê.

Bây giờ chúng ta đã biết một số điều về tốc độ tiệm cận mà sai số kiểm thử $\epsilon_\mathcal{D}(f)$ hội tụ về sai số thực tế $\epsilon(f)$, chúng ta có thể phóng to vào một số chi tiết quan trọng. Hãy nhớ rằng biến ngẫu nhiên quan tâm $\mathbf{1}(f(X) \neq Y)$ chỉ có thể nhận giá trị $0$ và $1$ và do đó là một biến ngẫu nhiên Bernoulli, được đặc trưng bởi một tham số chỉ ra xác suất nhận giá trị $1$. Ở đây, $1$ có nghĩa là bộ phân loại của chúng ta đã mắc lỗi, vì vậy tham số của biến ngẫu nhiên thực sự là tỷ lệ lỗi thực tế $\epsilon(f)$. Phương sai $\sigma^2$ của phân phối Bernoulli phụ thuộc vào tham số của nó (ở đây là $\epsilon(f)$) theo biểu thức $\epsilon(f)(1-\epsilon(f))$. Mặc dù ban đầu chưa biết $\epsilon(f)$, chúng ta biết rằng nó không thể lớn hơn $1$. Một nghiên cứu nhỏ về hàm này tiết lộ rằng phương sai của chúng ta là cao nhất khi tỷ lệ lỗi thực tế gần bằng $0.5$ và có thể thấp hơn nhiều khi nó gần bằng $0$ hoặc gần bằng $1$. Điều này cho chúng ta biết rằng độ lệch chuẩn tiệm cận của ước lượng $\epsilon_\mathcal{D}(f)$ về sai số $\epsilon(f)$ (trên lựa chọn $n$ mẫu kiểm thử) không thể lớn hơn $\sqrt{0.25/n}$.

Nếu chúng ta bỏ qua thực tế là tốc độ này đặc trưng cho hành vi khi kích thước tập kiểm thử tiến tới vô cùng thay vì khi chúng ta sở hữu các mẫu hữu hạn, điều này cho chúng ta biết rằng nếu chúng ta muốn sai số kiểm thử $\epsilon_\mathcal{D}(f)$ xấp xỉ sai số tổng thể $\epsilon(f)$ sao cho một độ lệch chuẩn tương ứng với khoảng $\pm 0.01$, thì chúng ta nên thu thập khoảng 2500 mẫu. Nếu chúng ta muốn khớp hai độ lệch chuẩn trong phạm vi đó và do đó tin tưởng 95% rằng $\epsilon_\mathcal{D}(f) \in \epsilon(f) \pm 0.01$, thì chúng ta sẽ cần 10,000 mẫu!

Đây hóa ra là kích thước của các tập kiểm thử cho nhiều điểm chuẩn (benchmarks) phổ biến trong học máy. Bạn có thể ngạc nhiên khi thấy rằng hàng nghìn bài báo học sâu ứng dụng được xuất bản mỗi năm tạo ra một tiếng vang lớn từ việc cải thiện tỷ lệ lỗi từ $0.01$ trở xuống. Tất nhiên, khi tỷ lệ lỗi gần bằng $0$ hơn nhiều, thì việc cải thiện $0.01$ thực sự có thể là một vấn đề lớn.

Một đặc điểm phiền toái trong phân tích của chúng ta cho đến nay là nó thực sự chỉ cho chúng ta biết về tiệm cận, nghĩa là mối quan hệ giữa $\epsilon_\mathcal{D}$ và $\epsilon$ phát triển như thế nào khi kích thước mẫu của chúng ta tiến tới vô cùng. Thật may mắn, vì biến ngẫu nhiên của chúng ta bị chặn, chúng ta có thể có được các giới hạn mẫu hữu hạn hợp lệ bằng cách áp dụng một bất đẳng thức của Hoeffding (1963):

$$P(\epsilon_\mathcal{D}(f) - \epsilon(f) \geq t) < \exp\left( - 2n t^2 \right).$$

Giải để tìm kích thước tập dữ liệu nhỏ nhất cho phép chúng ta kết luận với độ tin cậy 95% rằng khoảng cách $t$ giữa ước lượng $\epsilon_\mathcal{D}(f)$ và tỷ lệ lỗi thực tế $\epsilon(f)$ không vượt quá $0.01$, bạn sẽ thấy rằng cần khoảng 15,000 ví dụ so với 10,000 ví dụ được gợi ý bởi phân tích tiệm cận ở trên. Nếu bạn đi sâu hơn vào thống kê, bạn sẽ thấy rằng xu hướng này nói chung là đúng. Các đảm bảo giữ vững ngay cả trong các mẫu hữu hạn thường thận trọng hơn một chút. Lưu ý rằng trong cấu trúc của mọi thứ, các con số này không quá xa nhau, phản ánh tính hữu dụng chung của phân tích tiệm cận để cung cấp cho chúng ta các con số ước chừng ngay cả khi chúng không phải là các đảm bảo chúng ta có thể mang ra tòa.

## Tái Sử Dụng Tập Kiểm Thử (Test Set Reuse)

Theo một nghĩa nào đó, bây giờ bạn đã sẵn sàng để thành công trong việc tiến hành nghiên cứu học máy thực nghiệm. Gần như tất cả các mô hình thực tế đều được phát triển và xác thực dựa trên hiệu suất của tập kiểm thử và bây giờ bạn là một bậc thầy về tập kiểm thử. Đối với bất kỳ bộ phân loại cố định $f$ nào, bạn biết cách đánh giá sai số kiểm thử $\epsilon_\mathcal{D}(f)$ và biết chính xác những gì có thể (và không thể) nói về sai số tổng thể $\epsilon(f)$.

Vì vậy, hãy nói rằng bạn mang kiến thức này và chuẩn bị huấn luyện mô hình đầu tiên của mình $f_1$. Biết chính xác mức độ tự tin bạn cần có đối với hiệu suất tỷ lệ lỗi của bộ phân loại, bạn áp dụng phân tích của chúng ta ở trên để xác định số lượng ví dụ thích hợp cần đặt riêng cho tập kiểm thử. Hơn nữa, giả sử bạn đã khắc cốt ghi tâm các bài học từ :numref:`sec_generalization_basics` và đảm bảo giữ gìn tính thiêng liêng của tập kiểm thử bằng cách tiến hành tất cả các phân tích sơ bộ, tinh chỉnh siêu tham số và thậm chí lựa chọn giữa nhiều kiến trúc mô hình cạnh tranh trên một tập xác thực (validation set). Cuối cùng, bạn đánh giá mô hình $f_1$ trên tập kiểm thử và báo cáo một ước lượng không chệch về sai số tổng thể cùng với khoảng tin cậy liên quan.

Cho đến nay mọi thứ dường như đang diễn ra tốt đẹp. Tuy nhiên, đêm đó bạn thức dậy lúc 3 giờ sáng với một ý tưởng xuất chúng cho một phương pháp tiếp cận mô hình hóa mới. Ngày hôm sau, bạn lập trình mô hình mới của mình, tinh chỉnh các siêu tham số của nó trên tập xác thực và không những bạn giúp mô hình mới $f_2$ hoạt động thành công mà tỷ lệ lỗi của nó dường như thấp hơn nhiều so với $f_1$. Tuy nhiên, cảm giác hồi hộp của sự khám phá bỗng chốc biến mất khi bạn chuẩn bị cho việc đánh giá cuối cùng. Bạn không có tập kiểm thử!

Mặc dù tập kiểm thử gốc $\mathcal{D}$ vẫn đang nằm trên máy chủ của bạn, hiện tại bạn đang đối mặt với hai vấn đề đáng ngại. Đầu tiên, khi bạn thu thập tập kiểm thử của mình, bạn đã xác định mức độ chính xác cần thiết dưới giả định rằng bạn đang đánh giá một bộ phân loại duy nhất $f$. Tuy nhiên, nếu bạn bước vào công việc đánh giá nhiều bộ phân loại $f_1, ..., f_k$ trên cùng một tập kiểm thử, bạn phải xem xét vấn đề phát hiện sai (false discovery). Trước đây, bạn có thể chắc chắn 95% rằng $\epsilon_\mathcal{D}(f) \in \epsilon(f) \pm 0.01$ đối với một bộ phân loại duy nhất $f$ và do đó xác xuất của một kết quả sai lệch chỉ là 5%. Với $k$ bộ phân loại tham gia, có thể khó đảm bảo rằng không có dù chỉ một bộ phân loại trong số chúng có hiệu suất tập kiểm thử bị sai lệch. Với 20 bộ phân loại đang được xem xét, bạn có thể hoàn toàn không có khả năng loại trừ khả năng ít nhất một trong số chúng nhận được điểm số sai lệch. Vấn đề này liên quan đến thử nghiệm nhiều giả thuyết (multiple hypothesis testing), mặc dù có một kho tài liệu khổng lồ trong thống kê, vẫn là một vấn đề dai dẳng gây khó khăn cho nghiên cứu khoa học.

Nếu điều đó chưa đủ làm bạn lo lắng, thì có một lý do đặc biệt để không tin tưởng vào kết quả bạn nhận được trong các lần đánh giá tiếp theo. Hãy nhớ rằng phân tích của chúng ta về hiệu suất tập kiểm thử dựa trên giả định rằng bộ phân loại được chọn độc lập với tập kiểm thử và do đó chúng ta có thể coi tập kiểm thử được rút ra ngẫu nhiên từ tổng thể bên dưới. Ở đây, không những bạn đang thử nghiệm nhiều hàm, mà hàm tiếp theo $f_2$ đã được chọn sau khi bạn quan sát hiệu suất tập kiểm thử của $f_1$. Một khi thông tin từ tập kiểm thử đã rò rỉ đến người xây dựng mô hình, nó không bao giờ có thể là một tập kiểm thử thực sự theo nghĩa nghiêm ngặt nhất nữa. Vấn đề này được gọi là *quá khớp thích ứng* (adaptive overfitting) và gần đây đã nổi lên như một chủ đề được các nhà lý thuyết học tập và nhà thống kê đặc biệt quan tâm :cite:`dwork2015preserving`. May mắn thay, mặc dù có thể rò rỉ tất cả thông tin ra khỏi một tập giữ lại (holdout set), và các kịch bản tồi tệ nhất về mặt lý thuyết là rất ảm đạm, các phân tích này có thể quá thận trọng. Trong thực tế, hãy cẩn thận tạo ra các tập kiểm thử thực tế, tham khảo chúng càng ít càng tốt, tính đến việc thử nghiệm nhiều giả thuyết khi báo cáo các khoảng tin cậy và tăng cường cảnh giác tích cực hơn khi rủi ro cao và kích thước tập dữ liệu của bạn nhỏ. Khi chạy một loạt các thử thách điểm chuẩn, việc duy trì một vài tập kiểm thử thường là một thói quen tốt để sau mỗi vòng, tập kiểm thử cũ có thể được hạ cấp xuống thành tập xác thực.

## Lý Thuyết Học Thống Kê (Statistical Learning Theory)

Nói một cách đơn giản, *tập kiểm thử là tất cả những gì chúng ta thực sự có*, nhưng thực tế này có vẻ không thỏa mãn. Đầu tiên, chúng ta hiếm khi sở hữu một *tập kiểm thử thực sự*---trừ khi chúng ta là những người tạo ra tập dữ liệu, còn không thì người khác có thể đã đánh giá bộ phân loại của riêng họ trên tập dữ liệu được cho là "tập kiểm thử" của chúng ta rồi. Và ngay cả khi chúng ta là người đầu tiên có quyền truy cập, chúng ta sẽ sớm thấy mình thất vọng, ước gì có thể đánh giá các nỗ lực xây dựng mô hình tiếp theo của mình mà không có cảm giác dằn vặt rằng chúng ta không thể tin tưởng vào các con số của mình. Hơn nữa, ngay cả một tập kiểm thử thực sự cũng chỉ có thể cho chúng ta biết *post hoc* (hậu nghiệm) liệu một bộ phân loại có thực sự tổng quát hóa tốt cho tổng thể hay không, chứ không phải liệu chúng ta có bất kỳ lý do gì để kỳ vọng *a priori* (tiên nghiệm) rằng nó sẽ tổng quát hóa.

Với những mối lo ngại này trong tâm trí, bây giờ bạn có thể đã đủ hứng thú để thấy được sức hấp dẫn của *lý thuyết học thống kê*, một phân ngành toán học của học máy mà những người thực hành hướng tới việc làm sáng tỏ các nguyên tắc cơ bản giải thích tại sao/khi nào các mô hình được huấn luyện trên dữ liệu thực nghiệm có thể/sẽ tổng quát hóa cho dữ liệu chưa từng thấy. Một trong những mục tiêu chính của các nhà nghiên cứu học thống kê là giới hạn khoảng cách tổng quát hóa, liên hệ các thuộc tính của lớp mô hình với số lượng mẫu trong tập dữ liệu.

Các nhà lý thuyết học tập hướng tới việc giới hạn sự khác biệt giữa *sai số thực nghiệm* $\epsilon_\mathcal{S}(f_\mathcal{S})$ của một bộ phân loại đã học $f_\mathcal{S}$, cả hai đều được huấn luyện và đánh giá trên tập huấn luyện $\mathcal{S}$, và sai số thực tế $\epsilon(f_\mathcal{S})$ của chính bộ phân loại đó trên tổng thể bên dưới. Điều này có vẻ giống với bài toán đánh giá mà chúng ta vừa giải quyết nhưng có một sự khác biệt lớn. Trước đó, bộ phân loại $f$ đã cố định và chúng ta chỉ cần một tập dữ liệu cho mục đích đánh giá. Và quả thực, bất kỳ bộ phân loại cố định nào cũng tổng quát hóa: sai số của nó trên một tập dữ liệu (chưa từng thấy trước đó) là một ước lượng không chệch của sai số tổng thể. Nhưng chúng ta có thể nói gì khi một bộ phân loại được huấn luyện và đánh giá trên cùng một tập dữ liệu? Liệu chúng ta có thể tự tin rằng sai số huấn luyện sẽ gần với sai số kiểm thử?

Giả sử bộ phân loại đã học của chúng ta $f_\mathcal{S}$ phải được chọn từ một tập hợp các hàm được chỉ định trước $\mathcal{F}$. Hãy nhớ lại từ cuộc thảo luận của chúng ta về tập kiểm thử rằng mặc dù thật dễ dàng để ước lượng sai số của một bộ phân loại duy nhất, mọi thứ sẽ trở nên phức tạp khi chúng ta bắt đầu xem xét các tập hợp bộ phân loại. Ngay cả khi sai số thực nghiệm của bất kỳ bộ phân loại nào (cố định) sẽ gần với sai số thực tế của nó với xác suất cao, một khi chúng ta xem xét một tập hợp các bộ phân loại, chúng ta cần lo lắng về khả năng *chỉ một* trong số chúng sẽ nhận được một sai số ước lượng cực kỳ tệ. Mối lo ngại là chúng ta có thể chọn một bộ phân loại như vậy và do đó đánh giá thấp một cách nghiêm trọng sai số tổng thể. Hơn nữa, ngay cả đối với các mô hình tuyến tính, vì các tham số của chúng có giá trị liên tục, chúng ta thường chọn từ một lớp hàm vô hạn ($|\mathcal{F}| = \infty$).

Một giải pháp đầy tham vọng cho vấn đề này là phát triển các công cụ phân tích để chứng minh sự hội tụ đồng đều (uniform convergence), tức là với xác suất cao, tỷ lệ lỗi thực nghiệm cho mọi bộ phân loại trong lớp $f\in\mathcal{F}$ sẽ *đồng thời* hội tụ về tỷ lệ lỗi thực tế của nó. Nói cách khác, chúng ta tìm kiếm một nguyên lý lý thuyết cho phép chúng ta tuyên bố rằng với xác suất ít nhất $1-\delta$ (với một $\delta$ nhỏ nào đó) không có tỷ lệ lỗi của bộ phân loại nào $\epsilon(f)$ (trong số tất cả các bộ phân loại trong lớp $\mathcal{F}$) bị ước lượng sai lệch quá một lượng nhỏ $\alpha$. Rõ ràng, chúng ta không thể đưa ra những tuyên bố như vậy cho tất cả các lớp mô hình $\mathcal{F}$. Hãy nhớ lại lớp máy ghi nhớ luôn đạt được sai số thực nghiệm bằng $0$ nhưng không bao giờ vượt trội hơn việc đoán ngẫu nhiên trên tổng thể bên dưới.

Theo một nghĩa nào đó, lớp máy ghi nhớ quá linh hoạt. Không có kết quả hội tụ đồng đều nào có thể giữ vững. Mặt khác, một bộ phân loại cố định là vô dụng---nó tổng quát hóa hoàn hảo, nhưng không khớp với cả dữ liệu huấn luyện lẫn dữ liệu kiểm thử. Câu hỏi trung tâm của việc học do đó trong lịch sử đã được đóng khung như một sự đánh đổi giữa các lớp mô hình linh hoạt hơn (phương sai cao hơn) khớp tốt hơn với dữ liệu huấn luyện nhưng có nguy cơ quá khớp, so với các lớp mô hình cứng nhắc hơn (độ chệch cao hơn) tổng quát hóa tốt nhưng có nguy cơ dưới khớp. Một câu hỏi trung tâm trong lý thuyết học tập là phát triển phân tích toán học phù hợp để định lượng vị trí của một mô hình dọc theo phổ này và cung cấp các đảm bảo liên quan.

Trong một loạt các bài báo chuyên đề, Vapnik và Chervonenkis đã mở rộng lý thuyết về sự hội tụ của các tần suất tương đối sang các lớp hàm tổng quát hơn :cite:`VapChe64,VapChe68,VapChe71,VapChe74b,VapChe81,VapChe91`. Một trong những đóng góp chính của dòng nghiên cứu này là chiều Vapnik--Chervonenkis (chiều VC), đo lường (một khái niệm về) độ phức tạp (độ linh hoạt) của một lớp mô hình. Hơn nữa, một trong những kết quả chính của họ giới hạn sự khác biệt giữa sai số thực nghiệm và sai số tổng thể dưới dạng hàm của chiều VC và số lượng mẫu:

$$P\left(R[p, f] - R_\textrm{emp}[\mathbf{X}, \mathbf{Y}, f] < \alpha\right) \geq 1-\delta \ \textrm{ cho }\ \alpha \geq c \sqrt{(\textrm{VC} - \log \delta)/n}.$$

Ở đây $\delta > 0$ là xác suất giới hạn bị vi phạm, $\alpha$ là giới hạn trên của khoảng cách tổng quát hóa, và $n$ là kích thước tập dữ liệu. Cuối cùng, $c > 0$ là một hằng số chỉ phụ thuộc vào quy mô tổn thất có thể phát sinh. Một cách sử dụng giới hạn có thể là cắm vào các giá trị mong muốn của $\delta$ và $\alpha$ để xác định số lượng mẫu cần thu thập. Chiều VC định lượng số lượng điểm dữ liệu lớn nhất mà chúng ta có thể gán bất kỳ nhãn (nhị phân) tùy ý nào và đối với mỗi điểm dữ liệu tìm thấy một số mô hình $f$ trong lớp khớp với nhãn đó. Ví dụ, các mô hình tuyến tính trên đầu vào $d$ chiều có chiều VC là $d+1$. Thật dễ dàng để thấy rằng một đường thẳng có thể gán bất kỳ nhãn nào có thể cho ba điểm trong không gian hai chiều, nhưng không thể cho bốn điểm. Thật không may, lý thuyết này có xu hướng quá bi quan đối với các mô hình phức tạp hơn và việc đạt được đảm bảo này thường đòi hỏi nhiều ví dụ hơn thực tế cần thiết để đạt được tỷ lệ lỗi mong muốn. Lưu ý rằng khi cố định lớp mô hình và $\delta$, tỷ lệ lỗi của chúng ta lại suy giảm với tốc độ $\mathcal{O}(1/\sqrt{n})$ thông thường. Có vẻ như chúng ta không thể làm tốt hơn về mặt $n$. Tuy nhiên, khi chúng ta thay đổi lớp mô hình, chiều VC có thể đưa ra một bức tranh bi quan về khoảng cách tổng quát hóa.

## Tóm tắt

Cách đơn giản nhất để đánh giá một mô hình là tham khảo một tập kiểm thử bao gồm dữ liệu chưa từng thấy trước đó. Đánh giá tập kiểm thử cung cấp một ước lượng không chệch về sai số thực tế và hội tụ ở tốc độ mong muốn $\mathcal{O}(1/\sqrt{n})$ khi tập kiểm thử tăng lên. Chúng ta có thể cung cấp các khoảng tin cậy xấp xỉ dựa trên các phân phối tiệm cận chính xác hoặc các khoảng tin cậy mẫu hữu hạn hợp lệ dựa trên các đảm bảo mẫu hữu hạn (thận trọng hơn). Thực sự đánh giá tập kiểm thử là nền tảng của nghiên cứu học máy hiện đại. Tuy nhiên, các tập kiểm thử hiếm khi là các tập kiểm thử thực sự (được sử dụng lặp đi lặp lại bởi nhiều nhà nghiên cứu). Một khi cùng một tập kiểm thử được sử dụng để đánh giá nhiều mô hình, việc kiểm soát phát hiện sai có thể gặp khó khăn. Điều này có thể gây ra những vấn đề lớn về mặt lý thuyết. Trong thực tế, tầm quan trọng của vấn đề phụ thuộc vào kích thước của các tập giữ lại đang được đề cập và liệu chúng chỉ được sử dụng để chọn siêu tham số hay chúng đang rò rỉ thông tin trực tiếp hơn. Tuy nhiên, một thói quen tốt là quản lý các tập kiểm thử thực tế (hoặc nhiều tập) và càng thận trọng càng tốt về tần suất chúng được sử dụng.

Hy vọng cung cấp một giải pháp thỏa đáng hơn, các nhà lý thuyết học thống kê đã phát triển các phương pháp để đảm bảo sự hội tụ đồng đều trên một lớp mô hình. Nếu thực sự sai số thực nghiệm của mỗi mô hình đồng thời hội tụ về sai số thực tế của nó, thì chúng ta có thể tự do chọn mô hình hoạt động tốt nhất, giảm thiểu sai số huấn luyện, biết rằng nó cũng sẽ hoạt động tốt tương tự trên dữ liệu giữ lại. Cực kỳ quan trọng, bất kỳ kết quả nào trong số đó phải phụ thuộc vào một số thuộc tính của lớp mô hình. Vladimir Vapnik và Alexey Chernovenkis đã giới thiệu chiều VC, đưa ra kết quả hội tụ đồng đều áp dụng cho tất cả các mô hình trong một lớp VC. Sai số huấn luyện cho tất cả các mô hình trong lớp được đảm bảo (đồng thời) gần với sai số thực tế của chúng và được đảm bảo tiến gần hơn nữa ở tốc độ $\mathcal{O}(1/\sqrt{n})$. Sau phát hiện mang tính cách mạng về chiều VC, nhiều thước đo độ phức tạp thay thế đã được đề xuất, mỗi thước đo tạo điều kiện cho một đảm bảo tổng quát hóa tương tự. Xem :citet:`boucheron2005theory` để biết cuộc thảo luận chi tiết về một số cách tiên tiến để đo lường độ phức tạp của hàm. Thật không may, mặc dù các thước đo độ phức tạp này đã trở thành các công cụ hữu ích rộng rãi trong lý thuyết thống kê, chúng lại tỏ ra bất lực (khi được áp dụng trực tiếp) trong việc giải thích tại sao các mạng thần kinh sâu lại tổng quát hóa tốt. Các mạng thần kinh sâu thường có hàng triệu tham số (hoặc hơn) và có thể dễ dàng gán các nhãn ngẫu nhiên cho các tập hợp điểm lớn. Tuy nhiên, chúng tổng quát hóa tốt trên các bài toán thực tế và thật ngạc nhiên là chúng thường tổng quát hóa tốt hơn khi chúng lớn hơn và sâu hơn, mặc dù phải chịu chiều VC cao hơn. Trong chương tiếp theo, chúng ta sẽ xem xét lại khả năng tổng quát hóa trong bối cảnh học sâu.

## Bài tập

1. Nếu chúng ta muốn ước lượng sai số của một mô hình cố định $f$ trong khoảng $0.0001$ với xác suất lớn hơn 99.9%, chúng ta cần bao nhiêu mẫu?
2. Giả sử người khác sở hữu một tập kiểm thử được dán nhãn $\mathcal{D}$ và chỉ cung cấp các đầu vào chưa được dán nhãn (các đặc trưng). Bây giờ giả sử bạn chỉ có thể truy cập nhãn tập kiểm thử bằng cách chạy một mô hình $f$ (không có giới hạn nào được đặt ra cho lớp mô hình) trên mỗi đầu vào chưa được dán nhãn và nhận được sai số tương ứng $\epsilon_\mathcal{D}(f)$. Bạn cần đánh giá bao nhiêu mô hình trước khi rò rỉ toàn bộ tập kiểm thử và do đó có thể xuất hiện sai số bằng $0$, bất kể sai số thực tế của bạn là bao nhiêu?
3. Chiều VC của lớp đa thức bậc năm là bao nhiêu?
4. Chiều VC của các hình chữ nhật căn chỉnh theo trục (axis-aligned rectangles) trên dữ liệu hai chiều là bao nhiêu?

[Thảo luận](https://discuss.d2l.ai/t/6829)